In [2]:
# Librerias usadas 
import pandas as pd 
import openpyxl
import numpy as np
# para prueba jacque bera
from statsmodels.stats.stattools import jarque_bera
import matplotlib.pyplot as plt
from scipy.stats import norm

### **Rentabilidad del rendimiento semanal**

In [3]:
# Rentabilidad para un periodo de 5 dias 

# Rentabilidad para un periodo de tiempo t

df_acciones = pd.read_excel("Material_practico_2026-II.xlsx")
df_acciones

# Combertir JP Morgan a pesos Colombianos
df_acciones['J.P. Morgan Chase (COP)'] = round(df_acciones['J.P. Morgan Chase (USD)']*df_acciones['TRM (COP)'],0)
df_acciones


# Calcular (dato_actual - dato_anterior) / dato_anterior para todas las columnas
# Calculo de los rendimientos 

df_rendimiento = df_acciones.set_index('Fecha').sort_index()

# Calcular (dato_actual - dato_anterior) / dato_anterior para todas las columnas
columnas_precios = ['Bancolombia (P) (COP)', 'J.P. Morgan Chase (USD)', 'TRM (COP)', 'J.P. Morgan Chase (COP)']

for col in columnas_precios:
    df_rendimiento[f'Rend_{col}'] = np.log(df_rendimiento[col] / df_rendimiento[col].shift(1)).round(4) #np.log(df_rendimiento[col].pct_change().round(4))


df_rendimiento

,Bancolombia (P) (COP),J.P. Morgan Chase (USD),TRM (COP),J.P. Morgan Chase (COP),Rend_Bancolombia (P) (COP),Rend_J.P. Morgan Chase (USD),Rend_TRM (COP),Rend_J.P. Morgan Chase (COP)
Fecha,,,,,,,,
2015-01-02,16600.0,39.69,2284.22,90661.0,NaN,NaN,NaN,NaN
2015-01-03,17000.0,40.19,2284.22,91803.0,0.0238,0.0125,0.0000,0.0125
2015-01-04,16700.0,39.62,2282.27,90424.0,-0.0178,-0.0143,-0.0009,-0.0151
2015-01-05,17000.0,39.74,2282.13,90692.0,0.0178,0.0030,-0.0001,0.0030
2015-01-06,16900.0,40.02,2279.99,91245.0,-0.0059,0.0070,-0.0009,0.0061
...,...,...,...,...,...,...,...,...
2025-12-14,26900.0,84.76,2982.29,252779.0,-0.0148,0.0004,-0.0006,-0.0002
2025-12-15,27800.0,84.73,2964.56,251187.0,0.0329,-0.0004,-0.0060,-0.0063
2025-12-19,27240.0,84.94,2997.20,254582.0,-0.0203,0.0025,0.0109,0.0134


In [4]:
# Cantidad de dinero a invertir en cada acción
Cantidad_Banc = 1200
Cantidad_JP = 1000
Cantidad_TRM = 20000

# Precio de las acciones en la ultima fecha de cierre de la base de datos

precio_banc , precio_JP, precio_TRM = df_rendimiento.sort_values(by = "Fecha", ascending=False).head(1)['Bancolombia (P) (COP)'].iloc[0],\
df_rendimiento.sort_values(by = "Fecha", ascending=False).head(1)['J.P. Morgan Chase (COP)'].iloc[0],\
    df_rendimiento.sort_values(by = "Fecha", ascending=False).head(1)['TRM (COP)'].iloc[0]

# Pesos del portafolio 
w_banc = Cantidad_Banc*precio_banc
w_JP = Cantidad_JP*precio_JP
w_TRM = Cantidad_TRM*precio_TRM

portafolio = w_banc + w_JP + w_TRM

# % de pesos del portafolio
w_banc_prop = w_banc / portafolio *100
w_JP_prop = w_JP / portafolio *100
w_TRM_prop = w_TRM / portafolio *100

df_rendimiento.shape

(2474, 8)

In [5]:
print("Precio Bancolombia: ", precio_banc,"precio_JP: ", precio_JP, "precio_TRM: ", precio_TRM)
print("Pesos Bancolombia: ", w_banc,"Pesos JP: ", w_JP, "Pesos TRM: ", w_TRM)
print("pesos%% de Bancolombia: ", w_banc_prop,"pesos%% JP: ", w_JP_prop, "pesos%% TRM: ", w_TRM_prop)

Precio Bancolombia:  27460.0 precio_JP:  260764.0 precio_TRM:  2992.81
Pesos Bancolombia:  32952000.0 Pesos JP:  260764000.0 Pesos TRM:  59856200.0
pesos%% de Bancolombia:  9.319737241785411 pesos%% JP:  73.75127343156504 pesos%% TRM:  16.92898932664955


In [6]:
# Calculo del rendimeinto del portafolio
cols_rend = ['Rend_Bancolombia (P) (COP)', 'Rend_J.P. Morgan Chase (COP)','Rend_TRM (COP)']
pesos = [w_banc_prop, w_JP_prop, w_TRM_prop]

df_rendimiento['Rend_Portafolio'] = df_rendimiento[cols_rend].dot(pesos)/100
df_rendimiento

,Bancolombia (P) (COP),J.P. Morgan Chase (USD),TRM (COP),J.P. Morgan Chase (COP),Rend_Bancolombia (P) (COP),Rend_J.P. Morgan Chase (USD),Rend_TRM (COP),Rend_J.P. Morgan Chase (COP),Rend_Portafolio
Fecha,,,,,,,,,
2015-01-02,16600.0,39.69,2284.22,90661.0,NaN,NaN,NaN,NaN,NaN
2015-01-03,17000.0,40.19,2284.22,91803.0,0.0238,0.0125,0.0000,0.0125,0.011437
2015-01-04,16700.0,39.62,2282.27,90424.0,-0.0178,-0.0143,-0.0009,-0.0151,-0.012948
2015-01-05,17000.0,39.74,2282.13,90692.0,0.0178,0.0030,-0.0001,0.0030,0.003855
2015-01-06,16900.0,40.02,2279.99,91245.0,-0.0059,0.0070,-0.0009,0.0061,0.003797
...,...,...,...,...,...,...,...,...,...
2025-12-14,26900.0,84.76,2982.29,252779.0,-0.0148,0.0004,-0.0006,-0.0002,-0.001628
2025-12-15,27800.0,84.73,2964.56,251187.0,0.0329,-0.0004,-0.0060,-0.0063,-0.002596
2025-12-19,27240.0,84.94,2997.20,254582.0,-0.0203,0.0025,0.0109,0.0134,0.009836


In [7]:
# Calcular (dato_actual - dato_anterior) / dato_anterior para todas las columnas
columnas_precios = ['Bancolombia (P) (COP)', 'J.P. Morgan Chase (USD)', 'TRM (COP)', 'J.P. Morgan Chase (COP)']
df_rendimiento_5 = df_rendimiento.copy()

for col in columnas_precios:
    df_rendimiento_5[f'Rend_5{col}'] = np.log(df_rendimiento_5[col] / df_rendimiento_5[col].shift(4)).round(4) #np.log(df_rendimiento[col].pct_change().round(4))


df_rendimiento_5

,Bancolombia (P) (COP),J.P. Morgan Chase (USD),TRM (COP),J.P. Morgan Chase (COP),Rend_Bancolombia (P) (COP),Rend_J.P. Morgan Chase (USD),Rend_TRM (COP),Rend_J.P. Morgan Chase (COP),Rend_Portafolio,Rend_5Bancolombia (P) (COP),Rend_5J.P. Morgan Chase (USD),Rend_5TRM (COP),Rend_5J.P. Morgan Chase (COP)
Fecha,,,,,,,,,,,,,
2015-01-02,16600.0,39.69,2284.22,90661.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2015-01-03,17000.0,40.19,2284.22,91803.0,0.0238,0.0125,0.0000,0.0125,0.011437,NaN,NaN,NaN,NaN
2015-01-04,16700.0,39.62,2282.27,90424.0,-0.0178,-0.0143,-0.0009,-0.0151,-0.012948,NaN,NaN,NaN,NaN
2015-01-05,17000.0,39.74,2282.13,90692.0,0.0178,0.0030,-0.0001,0.0030,0.003855,NaN,NaN,NaN,NaN
2015-01-06,16900.0,40.02,2279.99,91245.0,-0.0059,0.0070,-0.0009,0.0061,0.003797,0.0179,0.0083,-0.0019,0.0064
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-12-14,26900.0,84.76,2982.29,252779.0,-0.0148,0.0004,-0.0006,-0.0002,-0.001628,0.0030,0.0127,-0.0111,0.0016
2025-12-15,27800.0,84.73,2964.56,251187.0,0.0329,-0.0004,-0.0060,-0.0063,-0.002596,0.0299,-0.0046,-0.0084,-0.0130
2025-12-19,27240.0,84.94,2997.20,254582.0,-0.0203,0.0025,0.0109,0.0134,0.009836,0.0088,-0.0065,-0.0019,-0.0083


In [9]:
# Calculo del rendimeinto del portafolio a 5 dias 
cols_rend = ['Rend_5Bancolombia (P) (COP)', 'Rend_5J.P. Morgan Chase (COP)','Rend_5TRM (COP)']
pesos = [w_banc_prop, w_JP_prop, w_TRM_prop]

df_rendimiento_5['Rend5_Portafolio'] = df_rendimiento_5[cols_rend].dot(pesos)/100
df_rendimiento_5

,Bancolombia (P) (COP),J.P. Morgan Chase (USD),TRM (COP),J.P. Morgan Chase (COP),Rend_Bancolombia (P) (COP),Rend_J.P. Morgan Chase (USD),Rend_TRM (COP),Rend_J.P. Morgan Chase (COP),Rend_Portafolio,Rend_5Bancolombia (P) (COP),Rend_5J.P. Morgan Chase (USD),Rend_5TRM (COP),Rend_5J.P. Morgan Chase (COP),Rend5_Portafolio
Fecha,,,,,,,,,,,,,,
2015-01-02,16600.0,39.69,2284.22,90661.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2015-01-03,17000.0,40.19,2284.22,91803.0,0.0238,0.0125,0.0000,0.0125,0.011437,NaN,NaN,NaN,NaN,NaN
2015-01-04,16700.0,39.62,2282.27,90424.0,-0.0178,-0.0143,-0.0009,-0.0151,-0.012948,NaN,NaN,NaN,NaN,NaN
2015-01-05,17000.0,39.74,2282.13,90692.0,0.0178,0.0030,-0.0001,0.0030,0.003855,NaN,NaN,NaN,NaN,NaN
2015-01-06,16900.0,40.02,2279.99,91245.0,-0.0059,0.0070,-0.0009,0.0061,0.003797,0.0179,0.0083,-0.0019,0.0064,0.006067
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-12-14,26900.0,84.76,2982.29,252779.0,-0.0148,0.0004,-0.0006,-0.0002,-0.001628,0.0030,0.0127,-0.0111,0.0016,-0.000420
2025-12-15,27800.0,84.73,2964.56,251187.0,0.0329,-0.0004,-0.0060,-0.0063,-0.002596,0.0299,-0.0046,-0.0084,-0.0130,-0.008223
2025-12-19,27240.0,84.94,2997.20,254582.0,-0.0203,0.0025,0.0109,0.0134,0.009836,0.0088,-0.0065,-0.0019,-0.0083,-0.005623


In [10]:
# Teorico

Rentabilidad_diaria = np.mean(df_rendimiento_5['Rend5_Portafolio'].dropna())
volatiliad_diaria = np.std(df_rendimiento_5['Rend5_Portafolio'].dropna())
rentabilidad = Rentabilidad_diaria * 5
volatilidad = volatiliad_diaria * np.sqrt(5)
varianza = volatilidad**2

In [11]:
# Real

rentabilidad_real = np.mean(df_rendimiento_5['Rend5_Portafolio'].dropna())
volatilidad_real = np.std(df_rendimiento_5['Rend5_Portafolio'].dropna())
varianza_real = volatilidad_real**2

In [12]:
print("Teorico")
print("Rentabilidad diaria: ", Rentabilidad_diaria)
print("Volatilidad diaria: ", volatiliad_diaria*100)
print("")
print("Rentabilidad para 20 días: ", rentabilidad*100)
print("Volatilidad para 20 días: ", volatilidad*100)
print("Varianza para 20 días: ", varianza*100)

print("")
print("Real")
print("Rentabilidad diaria: ", rentabilidad_real*100)   
print("Volatilidad diaria: ", volatilidad_real*100)
print("Varianza diaria: ", varianza_real*100)

Teorico
Rentabilidad diaria:  0.0013868297751448834
Volatilidad diaria:  3.91667823684472

Rentabilidad para 20 días:  0.6934148875724416
Volatilidad para 20 días:  8.757958783578816
Varianza para 20 días:  0.7670184205486534

Real
Rentabilidad diaria:  0.13868297751448835
Volatilidad diaria:  3.91667823684472
Varianza diaria:  0.15340368410973065
